In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
! pip install ultralytics
! pip install huggingface_hub
! pip install roboflow

## Imports

In [7]:
from ultralytics import YOLO
from roboflow import Roboflow
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## Subtask 1: Fire and Smoke Detection Using YOLO

In [8]:
from roboflow import Roboflow
rf = Roboflow(api_key="w06Mz6QITM0P0KlSycgf")
project = rf.workspace("gfo-fc1de").project("fire-and-smoke-detector-yykya")
version = project.version(14)
dataset = version.download("yolov8")



loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to fire-and-smoke-detector-14 in yolov8:: 100%|██████████| 11204/11204 [00:00<00:00, 11407.89it/s]


In [9]:
data_yaml = os.path.join(dataset.location, "data.yaml")

## Understanding YOLO Dataset Format

YOLO format contains images and text annotation files.

Each annotation file contains:
- Class ID
- Object center X
- Object center Y
- Bounding box width
- Bounding box height

All coordinates are normalized between 0 and 1.

### data.yaml

The data.yaml file contains:
- Dataset paths
- Number of classes
- Class names

### Dataset Splits

- Train: Used to train the model.
- Validation: Used to evaluate the model during training.
- Test: Used for final model evaluation.

These splits help measure how well the model performs on unseen images.

In [ ]:
model = YOLO("yolov8n.pt")
result = model.train(
    data = data_yaml,
    epochs = 50,
    imgsz = 640,
    batch = 8,
    name="yolov8_fire_smoke"
)

In [12]:
save_dir = result.save_dir
print("Training output folder:", save_dir)
print("Files:", os.listdir(save_dir))
print("best.pt:", os.path.join(save_dir, "weights", "best.pt"))
print("last.pt:", os.path.join(save_dir, "weights", "last.pt"))

Training output folder: /kaggle/working/runs/detect/yolov8_fire_smoke-2
Files: ['BoxF1_curve.png', 'confusion_matrix.png', 'results.csv', 'train_batch19720.jpg', 'train_batch0.jpg', 'args.yaml', 'BoxR_curve.png', 'train_batch19721.jpg', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg', 'val_batch1_labels.jpg', 'train_batch2.jpg', 'train_batch1.jpg', 'BoxP_curve.png', 'val_batch2_pred.jpg', 'val_batch0_labels.jpg', 'results.png', 'BoxPR_curve.png', 'val_batch1_pred.jpg', 'val_batch2_labels.jpg', 'labels.jpg', 'train_batch19722.jpg', 'weights']
best.pt: /kaggle/working/runs/detect/yolov8_fire_smoke-2/weights/best.pt
last.pt: /kaggle/working/runs/detect/yolov8_fire_smoke-2/weights/last.pt


In [ ]:
best_model = YOLO(os.path.join(save_dir, "weights", "best.pt"))

video_path = "/kaggle/input/datasets/rehammohamedramadan/firevideo/fire.mp4"  
pred_results = best_model.predict(
    source=video_path,
    conf=0.35,
    save=True,
    project="runs/predict",
    name="yolov8_fire_video"
)

print("Output saved at:", pred_results[0].save_dir)

## Understanding YOLO Training

### Object Detection
Object detection means identifying objects and locating them
using bounding boxes.

### Why YOLO Nano?
YOLO Nano is a lightweight model. It requires fewer resources
and provides faster training and inference.

### Epochs
An epoch means one complete pass through the training dataset.

### Image Size
Image size represents the dimensions used to process training images.
We use 640 pixels to balance speed and detection accuracy.

### What Happens During Training?
The model learns object features from labeled images.
It updates its weights to reduce prediction errors.